Import libraries

In [ ]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [ ]:
%time train_df = pd.read_csv(r"/benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/val_men.csv")

In [ ]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [ ]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [ ]:
print('X_train[:10]', X_train[:1].astype(float))

In [ ]:
print('y_train[:10]', y_train[:1].astype(float))

In [ ]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

In [ ]:
epochs = 70
lr = 5e-5
wd = 1e-5
alpha = 1.0
beta = 1.0
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 15
shared_dropout = 0
outcome_droupout = 0 
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import io
from contextlib import redirect_stdout, redirect_stderr
from itertools import product

# 1. Grid search config
seeds = [412312, 42, 1874, 902745, 1]
lr_grid = [5e-5, 1e-4, 5e-4]
wd_grid = [0.0, 1e-6, 1e-5, 1e-4]
alpha_grid = [0.1, 0.5, 1.0]
beta_grid = [0.1, 0.5, 1.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
grid_results = []

# 2. Loop over all (lr, wd, alpha, beta) pairs
for grid_lr, grid_wd, grid_alpha, grid_beta in product(lr_grid, wd_grid, alpha_grid, beta_grid):
    run_rows = []

    for SEED in seeds:
        seed_everything(SEED)

        # Reinitialize model for each seed
        dragonnet = Dragonnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            beta=grid_beta,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_droupout=outcome_droupout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence verbose training logs; keep only final aggregated results.
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            dragonnet.fit(train_loader, val_loader)

        # Validation prediction
        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p, _, _ = dragonnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

        # Test prediction
        x_test_device = x_men_test_t.to(device)
        y0_pred, y1_pred, _, _ = dragonnet.predict(x_test_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_test_t.detach().cpu().numpy().flatten()
        t_true = t_men_test_t.detach().cpu().numpy().flatten()

        # ATE error
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

        row = {
            "Seed": SEED,
            "lr": grid_lr,
            "weight_decay": grid_wd,
            "alpha": grid_alpha,
            "beta": grid_beta,
            "Val_AUQC": current_val_auqc,
            "AUUC": auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "AUQC": auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "Lift": lift(y_true, t_true, uplift_pred, h=0.3),
            "KRCC": krcc(y_true, t_true, uplift_pred, bins=100),
            "ATE_Err": abs(ate_pred - ate_true)
        }
        run_rows.append(row)

    # Aggregate each hyperparameter pair
    df_pair = pd.DataFrame(run_rows)
    mean_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].mean()
    std_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].std()

    grid_results.append({
        "lr": grid_lr,
        "weight_decay": grid_wd,
        "alpha": grid_alpha,
        "beta": grid_beta,
        "mean_Val_AUQC": mean_pair["Val_AUQC"],
        "mean_AUUC": mean_pair["AUUC"],
        "mean_AUQC": mean_pair["AUQC"],
        "mean_Lift": mean_pair["Lift"],
        "mean_KRCC": mean_pair["KRCC"],
        "mean_ATE_Err": mean_pair["ATE_Err"],
        "std_Val_AUQC": std_pair["Val_AUQC"],
        "std_AUUC": std_pair["AUUC"],
        "std_AUQC": std_pair["AUQC"],
        "std_Lift": std_pair["Lift"],
        "std_KRCC": std_pair["KRCC"],
        "std_ATE_Err": std_pair["ATE_Err"]
    })

# 3. Final ranking by mean test AUQC
df_grid = pd.DataFrame(grid_results).sort_values("mean_AUQC", ascending=False).reset_index(drop=True)
best_cfg = df_grid.iloc[0]

print("\n" + "=" * 130)
print(f"{'KẾT QUẢ GRID SEARCH (XẾP HẠNG THEO MEAN TEST AUQC)':^130}")
print("=" * 130)

display_cols = [
    "lr", "weight_decay", "alpha", "beta",
    "mean_Val_AUQC", "mean_AUUC", "mean_AUQC", "mean_Lift", "mean_KRCC", "mean_ATE_Err",
    "std_Val_AUQC", "std_AUUC", "std_AUQC", "std_Lift", "std_KRCC", "std_ATE_Err"
]

print(df_grid[display_cols].to_string(
    index=False,
    formatters={
        "lr": "{:.1e}".format,
        "weight_decay": "{:.1e}".format,
        "alpha": "{:.1f}".format,
        "beta": "{:.1f}".format,
        "mean_Val_AUQC": "{:.4f}".format,
        "mean_AUUC": "{:.4f}".format,
        "mean_AUQC": "{:.4f}".format,
        "mean_Lift": "{:.4f}".format,
        "mean_KRCC": "{:.4f}".format,
        "mean_ATE_Err": "{:.4f}".format,
        "std_Val_AUQC": "{:.4f}".format,
        "std_AUUC": "{:.4f}".format,
        "std_AUQC": "{:.4f}".format,
        "std_Lift": "{:.4f}".format,
        "std_KRCC": "{:.4f}".format,
        "std_ATE_Err": "{:.4f}".format
    }
))

print("-" * 130)
print("Best hyperparameters by mean TEST AUQC:")
print(
    f"  lr={best_cfg['lr']:.1e}, weight_decay={best_cfg['weight_decay']:.1e}, "
    f"alpha={best_cfg['alpha']:.1f}, beta={best_cfg['beta']:.1f}"
)
print(f"  mean TEST AUQC = {best_cfg['mean_AUQC']:.4f} ± {best_cfg['std_AUQC']:.4f}")
print("=" * 130)

In [ ]:
# import pandas as pd
# import numpy as np
# import torch

# # 1. Cấu hình
# seeds = [412312, 42, 1874, 902745, 1]
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# all_runs = []

# print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# # 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
# for SEED in seeds:
#     seed_everything(SEED)
    
#     # Khởi tạo lại mô hình
#     dragonnet = Dragonnet(
#         input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr,
#         alpha=alpha, beta=beta,
#         weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
#         shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
#         outcome_droupout=outcome_droupout, shared_dropout=shared_dropout,
#         early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
#         activation=activation
#     )
    
#     dragonnet.fit(train_loader, val_loader)
    
#     # Predict
#     x_men_test_t_on_device = x_men_test_t.to(device)
#     y0_pred, y1_pred, _, _ = dragonnet.predict(x_men_test_t_on_device)
    
#     uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
#     y_true = y_men_test_t.cpu().numpy().flatten()
#     t_true = t_men_test_t.cpu().numpy().flatten()
    
#     # Tính toán
#     ate_pred = uplift_pred.mean()
#     ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
#     all_runs.append({
#         'Seed': SEED,
#         'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
#         'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
#         'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
#         'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
#         'ATE_Err': abs(ate_pred - ate_true)
#     })
#     print(f"  ✔️ Đã xong Seed {SEED}")

# # 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
# df_results = pd.DataFrame(all_runs)

# print("\n" + "="*85)
# print(f"{'CHI TIẾT TỪNG SEED':^85}")
# print("="*85)
# # Sử dụng to_string để in bảng đẹp mắt
# print(df_results.to_string(index=False, formatters={
#     'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
#     'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
# }))

# # 4. Tính toán Mean và Std
# mean_res = df_results.drop(columns='Seed').mean()
# std_res = df_results.drop(columns='Seed').std()

# print("="*85)
# print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
# print("-" * 85)
# summary_data = []
# for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
#     print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

# print("="*85)